In [ ]:
import torch
import torchvision

print("🟢 PyTorch version:", torch.__version__)
print("🟢 Torchvision version:", torchvision.__version__)
print("🟢 CUDA available:", torch.cuda.is_available())
print("🟢 CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

try:
    import detectron2
    print("🟢 Detectron2 module is importable.")
except ImportError:
    print("🔴 Detectron2 is NOT installed.")


In [ ]:
!python -m pip install pyyaml==5.1
import sys, os, distutils.core
# Note: This is a faster way to install detectron2 in Colab, but it does not include all functionalities (e.g. compiled operators).
# See https://detectron2.readthedocs.io/tutorials/install.html for full installation instructions
!git clone 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install {' '.join([f"'{x}'" for x in dist.install_requires])}
sys.path.insert(0, os.path.abspath('./detectron2'))

# Properly install detectron2. (Please do not install twice in both ways)
# !python -m pip install 'git+https://github.com/facebookresearch/detectron2.git'

In [3]:
import torch, detectron2
!nvcc --version
TORCH_VERSION = ".".join(torch.__version__.split(".")[:2])
CUDA_VERSION = torch.__version__.split("+")[-1]
print("torch: ", TORCH_VERSION, "; cuda: ", CUDA_VERSION)
print("detectron2:", detectron2.__version__)

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0
torch:  2.6 ; cuda:  cu124
detectron2: 0.6


In [4]:
# Some basic setup:
# Setup detectron2 logger
import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

# import some common libraries
import numpy as np
import os, json, cv2, random
from google.colab.patches import cv2_imshow

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg
from detectron2.utils.visualizer import Visualizer
from detectron2.data import MetadataCatalog, DatasetCatalog

In [5]:
from detectron2.data.datasets import register_coco_instances

register_coco_instances(
    "train_set", {}, 
    "/kaggle/input/funnyann/annotations_coco_train.json", 
    "/kaggle/input/d/meteharunakcay/visual-redactions/images/train"
)


register_coco_instances(
    "val_set", {}, 
    "/kaggle/input/funnyann/annotations_coco_val.json", 
    "/kaggle/input/d/meteharunakcay/visual-redactions/images/val"
)


In [6]:
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer
import os
import shutil

cfg = get_cfg()

# Load config
cfg.merge_from_file(model_zoo.get_config_file(
    "COCO-InstanceSegmentation/mask_rcnn_X_101_32x8d_FPN_3x.yaml"
))

cfg.DATASETS.TRAIN = ("train_set",)
cfg.DATASETS.TEST = ("val_set",)

cfg.OUTPUT_DIR = "./output_detectron2"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# Copy previous checkpoint files to new output directory
shutil.copy("/kaggle/input/visual-model3/last_checkpoint", cfg.OUTPUT_DIR)
shutil.copy("/kaggle/input/visual-model3/model_0029999.pth", os.path.join(cfg.OUTPUT_DIR, "model_final.pth"))

# Training params
cfg.DATALOADER.NUM_WORKERS = 2
cfg.SOLVER.IMS_PER_BATCH = 2
cfg.SOLVER.BASE_LR = 0.00025
cfg.SOLVER.MAX_ITER = 40000
cfg.TEST.EVAL_PERIOD = 10000
cfg.SOLVER.STEPS = []
cfg.SOLVER.CHECKPOINT_PERIOD = 10000

cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = 128
cfg.MODEL.ROI_HEADS.NUM_CLASSES = 9

In [7]:
from detectron2.engine import DefaultTrainer

trainer = DefaultTrainer(cfg)
trainer.resume_or_load(resume=True)
trainer.train()


[07/29 06:40:14 d2.engine.defaults]: Model:
GeneralizedRCNN(
  (backbone): FPN(
    (fpn_lateral2): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output2): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelMaxPool()
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res

/usr/local/lib/python3.11/dist-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[07/29 06:41:03 d2.utils.events]:  eta: 6:08:13  iter: 30019  total_loss: 0.337  loss_cls: 0.05393  loss_box_reg: 0.1252  loss_mask: 0.1114  loss_rpn_cls: 0.0003147  loss_rpn_loc: 0.01249    time: 2.1604  last_time: 2.3830  data_time: 0.0590  last_data_time: 0.0041   lr: 0.00025  max_mem: 5711M


2025-07-29 06:41:05.922353: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753771266.091726      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753771266.139896      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


[07/29 06:42:00 d2.utils.events]:  eta: 6:18:45  iter: 30039  total_loss: 0.2736  loss_cls: 0.04009  loss_box_reg: 0.1243  loss_mask: 0.1007  loss_rpn_cls: 0.0004256  loss_rpn_loc: 0.01466    time: 2.1824  last_time: 2.7598  data_time: 0.0053  last_data_time: 0.0055   lr: 0.00025  max_mem: 5711M
[07/29 06:42:51 d2.utils.events]:  eta: 6:25:02  iter: 30059  total_loss: 0.3056  loss_cls: 0.0523  loss_box_reg: 0.1268  loss_mask: 0.1163  loss_rpn_cls: 0.000784  loss_rpn_loc: 0.01459    time: 2.3082  last_time: 2.9066  data_time: 0.0054  last_data_time: 0.0053   lr: 0.00025  max_mem: 5711M
[07/29 06:43:39 d2.utils.events]:  eta: 6:29:57  iter: 30079  total_loss: 0.2913  loss_cls: 0.03716  loss_box_reg: 0.1309  loss_mask: 0.111  loss_rpn_cls: 0.0003242  loss_rpn_loc: 0.01279    time: 2.3224  last_time: 2.5570  data_time: 0.0055  last_data_time: 0.0076   lr: 0.00025  max_mem: 5711M
[07/29 06:44:28 d2.utils.events]:  eta: 6:33:21  iter: 30099  total_loss: 0.3851  loss_cls: 0.06166  loss_box_re

In [7]:
register_coco_instances(
    "test",
    {},
    "/kaggle/input/funnyann/annotations_coco_test.json",  # path to annotations
    "/kaggle/input/d/meteharunakcay/visual-redactions/images/test"       # image directory
)

In [8]:
import os
import numpy as np
import cv2
from tqdm import tqdm
from pycocotools.coco import COCO
from detectron2.engine import DefaultPredictor
from detectron2.data.datasets import register_coco_instances
from detectron2.utils.visualizer import GenericMask
from sklearn.metrics import confusion_matrix

coco = COCO("/kaggle/input/funnyann/annotations_coco_test.json")
img_dir = "/kaggle/input/d/meteharunakcay/visual-redactions/images/test"

# -------------------------------
# Step 2: Predictor Setup
# -------------------------------
cfg.MODEL.WEIGHTS = "/kaggle/input/visual-model4/model_0039999.pth"
cfg.DATASETS.TEST = ("test",)
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5

predictor = DefaultPredictor(cfg)

# -------------------------------
# Step 3: Utility - IoU
# -------------------------------
def compute_iou(mask1, mask2):
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    if union == 0:
        return np.nan
    return intersection / union

# -------------------------------
# Step 4: mIoU Per Class
# -------------------------------
from collections import defaultdict

# Map category_id (1–9) to internal index (0–8)
category_id_to_index = {cat["id"]: idx for idx, cat in enumerate(coco.loadCats(coco.getCatIds()))}
class_index_to_name = {idx: cat["name"] for idx, cat in enumerate(coco.loadCats(coco.getCatIds()))}

iou_per_class = defaultdict(list)

for img_id in tqdm(coco.getImgIds()):
    img_info = coco.loadImgs([img_id])[0]
    img_path = os.path.join(img_dir, img_info['file_name'])
    image = cv2.imread(img_path)

    # Ground Truth Masks (mapped to 0–8)
    ann_ids = coco.getAnnIds(imgIds=[img_id])
    anns = coco.loadAnns(ann_ids)

    gt_masks = defaultdict(lambda: np.zeros(image.shape[:2], dtype=np.uint8))
    for ann in anns:
        cat_id = ann['category_id']
        class_index = category_id_to_index[cat_id]  # map 1–9 → 0–8
        mask = coco.annToMask(ann)
        gt_masks[class_index] = np.logical_or(gt_masks[class_index], mask)

    # Predicted Masks (already 0–8)
    outputs = predictor(image)
    instances = outputs["instances"].to("cpu")

    pred_masks = defaultdict(lambda: np.zeros(image.shape[:2], dtype=np.uint8))
    for i in range(len(instances)):
        pred_class = int(instances.pred_classes[i])
        pred_mask = instances.pred_masks[i].numpy()
        pred_masks[pred_class] = np.logical_or(pred_masks[pred_class], pred_mask)

    # Compute IoU
    all_class_indices = set(list(gt_masks.keys()) + list(pred_masks.keys()))
    for class_idx in all_class_indices:
        iou = compute_iou(gt_masks[class_idx], pred_masks[class_idx])
        if not np.isnan(iou):
            iou_per_class[class_idx].append(iou)


# -------------------------------
# Step 5: Report mIoU per Class
# -------------------------------
print("\nClass-wise mIoU:")
for class_id, ious in sorted(iou_per_class.items()):
    name = class_index_to_name.get(class_id, f"Unknown_{class_id}")
    miou = np.mean(ious)
    print(f"{name} (ID {class_id}): mIoU = {miou:.4f}")




loading annotations into memory...
Done (t=0.33s)
creating index...
index created!
[07/30 06:25:51 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/input/visual-model4/model_0039999.pth ...


  0%|          | 0/2989 [00:00<?, ?it/s]/usr/local/lib/python3.11/dist-packages/torch/functional.py:539: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:3637.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
100%|██████████| 2989/2989 [26:28<00:00,  1.88it/s]


Class-wise mIoU:
a105_face_all (ID 0): mIoU = 0.7407
a108_license_plate_all (ID 1): mIoU = 0.5325
a109_person_body (ID 2): mIoU = 0.8072
a110_nudity_all (ID 3): mIoU = 0.3456
a26_handwriting (ID 4): mIoU = 0.3274
a39_disability_physical (ID 5): mIoU = 0.2487
a43_medicine (ID 6): mIoU = 0.1112
a7_fingerprint (ID 7): mIoU = 0.3759
a8_signature (ID 8): mIoU = 0.2644


In [9]:
# --- Add background to class mappings ---
class_index_to_name = {0: "background"}
category_id_to_index = {}
for idx, cat in enumerate(coco.loadCats(coco.getCatIds()), start=1):
    category_id_to_index[cat["id"]] = idx
    class_index_to_name[idx] = cat["name"]

# --- mIoU Calculation ---
iou_per_class = defaultdict(list)

for img_id in tqdm(coco.getImgIds()):
    img_info = coco.loadImgs([img_id])[0]
    img_path = os.path.join(img_dir, img_info['file_name'])
    image = cv2.imread(img_path)
    height, width = image.shape[:2]

    # Ground Truth Masks
    ann_ids = coco.getAnnIds(imgIds=[img_id])
    anns = coco.loadAnns(ann_ids)

    gt_combined = np.zeros((height, width), dtype=np.uint8)
    gt_masks = defaultdict(lambda: np.zeros((height, width), dtype=np.uint8))

    for ann in anns:
        cat_id = ann['category_id']
        class_index = category_id_to_index[cat_id]
        mask = coco.annToMask(ann)
        gt_masks[class_index] = np.logical_or(gt_masks[class_index], mask)
        gt_combined = np.logical_or(gt_combined, mask)

    gt_masks[0] = np.logical_not(gt_combined)  # background = areas not covered by any annotation

    # Prediction Masks
    outputs = predictor(image)
    instances = outputs["instances"].to("cpu")

    pred_combined = np.zeros((height, width), dtype=np.uint8)
    pred_masks = defaultdict(lambda: np.zeros((height, width), dtype=np.uint8))

    for i in range(len(instances)):
        pred_class = int(instances.pred_classes[i]) + 1  # shift: model classes 0–8 → 1–9
        pred_mask = instances.pred_masks[i].numpy()
        pred_masks[pred_class] = np.logical_or(pred_masks[pred_class], pred_mask)
        pred_combined = np.logical_or(pred_combined, pred_mask)

    pred_masks[0] = np.logical_not(pred_combined)  # background

    # Compute IoUs
    all_class_indices = set(gt_masks.keys()).union(set(pred_masks.keys()))
    for class_idx in all_class_indices:
        iou = compute_iou(gt_masks[class_idx], pred_masks[class_idx])
        if not np.isnan(iou):
            iou_per_class[class_idx].append(iou)

# --- Final Reporting ---
print("\nClass-wise mIoU:")
for class_id in sorted(iou_per_class.keys()):
    name = class_index_to_name.get(class_id, f"Unknown_{class_id}")
    miou = np.mean(iou_per_class[class_id])
    print(f"{name} (ID {class_id}): mIoU = {miou:.4f}")


100%|██████████| 2989/2989 [28:21<00:00,  1.76it/s]


Class-wise mIoU:
background (ID 0): mIoU = 0.9108
a105_face_all (ID 1): mIoU = 0.7407
a108_license_plate_all (ID 2): mIoU = 0.5325
a109_person_body (ID 3): mIoU = 0.8072
a110_nudity_all (ID 4): mIoU = 0.3456
a26_handwriting (ID 5): mIoU = 0.3274
a39_disability_physical (ID 6): mIoU = 0.2487
a43_medicine (ID 7): mIoU = 0.1112
a7_fingerprint (ID 8): mIoU = 0.3759
a8_signature (ID 9): mIoU = 0.2644
